In [2]:
# ── UI パネル（パラメータ調整） ────────────────────

!pip install -q ipywidgets nest-asyncio edge-tts

from ipywidgets import IntSlider, Textarea, Dropdown, Button, VBox, HBox, Label, Output
from IPython.display import display, clear_output, Audio
import ipywidgets as widgets
import nest_asyncio
import asyncio
import datetime
import os
import edge_tts

nest_asyncio.apply()

LOCAL_AUDIO_DIR = "/content/audio_output"
os.makedirs(LOCAL_AUDIO_DIR, exist_ok=True)

VOICE_PRIORITY = [
    "ja-JP-NanamiNeural",
    "ja-JP-KeitaNeural",
]

# ═══════════════════════════════════════════════════
# 📝 入力フィールド
# ═══════════════════════════════════════════════════

text_input = Textarea(
    value="こんにちは。今日はどんなお話をしましょうか？もしよければ、ゆっくりお話ししますね。",
    placeholder="読み上げるテキストを入力してください",
    description="テキスト:",
    rows=4,
    style={'description_width': '80px'}
)

voice_dropdown = Dropdown(
    options=VOICE_PRIORITY,
    value=VOICE_PRIORITY[0],
    description="音声:",
    style={'description_width': '80px'}
)

# ═══════════════════════════════════════════════════
# 🎚️  パラメータスライダー
# ═══════════════════════════════════════════════════

rate_slider = IntSlider(
    value=-5,
    min=-50,
    max=50,
    step=1,
    description="速度（Rate）:",
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)
rate_label = Label(value="-5%")

pitch_slider = IntSlider(
    value=10,
    min=-50,
    max=50,
    step=1,
    description="ピッチ:",
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)
pitch_label = Label(value="+10Hz")

volume_slider = IntSlider(
    value=0,
    min=-100,
    max=100,
    step=1,
    description="ボリューム:",
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)
volume_label = Label(value="+0%")

# スライダー値の変更時にラベルを更新
def update_labels(change):
    rate_label.value = f"{rate_slider.value:+d}%"
    pitch_label.value = f"{pitch_slider.value:+d}Hz"
    volume_label.value = f"{volume_slider.value:+d}%"

rate_slider.observe(update_labels, 'value')
pitch_slider.observe(update_labels, 'value')
volume_slider.observe(update_labels, 'value')

# ═══════════════════════════════════════════════════
# 🔘 実行ボタン
# ═══════════════════════════════════════════

execute_button = Button(
    description="🎵 実行（合成・再生）",
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

output_area = Output()

async def synthesize_with_ui():
    """UI パラメータで音声合成実行"""
    text = text_input.value
    voice = voice_dropdown.value
    rate_val = rate_slider.value
    pitch_val = pitch_slider.value
    volume_val = volume_slider.value
    
    output_area.clear_output()
    with output_area:
        ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = f"{LOCAL_AUDIO_DIR}/edge_{ts}.mp3"
        
        print(f"⏳ 処理中...")
        print(f"  テキスト: {text[:40]}...")
        print(f"  音声: {voice}")
        print(f"  Rate: {rate_val:+d}%, Pitch: {pitch_val:+d}Hz, Volume: {volume_val:+d}%")
        
        try:
            params = {}
            if rate_val != 0:
                params['rate'] = f"{rate_val:+d}%"
            if pitch_val != 0:
                params['pitch'] = f"{pitch_val:+d}Hz"
            if volume_val != 0:
                params['volume'] = f"{volume_val:+d}%"
            
            communicate = edge_tts.Communicate(
                text,
                voice=voice,
                **params,
            )
            await communicate.save(output_path)
            
            print(f"\n✅ 完了！")
            print(f"📁 保存: {output_path}")
            print()
            display(Audio(output_path, autoplay=False))
            
        except Exception as e:
            print(f"\n❌ エラーが発生しました: {type(e).__name__}")
            print(f"   {e}")

def on_execute_clicked(button):
    """ボタンクリック時の処理"""
    asyncio.run(synthesize_with_ui())

execute_button.on_click(on_execute_clicked)

# ═══════════════════════════════════════════════════
# 🎨 UI レイアウト
# ═══════════════════════════════════════════

settings_box = VBox([
    Label(value="⚙️ 音声パラメータ調整", style={'font_weight': 'bold'}),
    HBox([rate_slider, rate_label]),
    HBox([pitch_slider, pitch_label]),
    HBox([volume_slider, volume_label]),
])

input_box = VBox([
    Label(value="📋 入力設定", style={'font_weight': 'bold'}),
    HBox([Label(value="テキスト:"), text_input]),
    HBox([Label(value="音声:"), voice_dropdown]),
])

control_panel = VBox([
    input_box,
    settings_box,
    execute_button,
    output_area
], layout=widgets.Layout(
    border='1px solid #ddd',
    padding='15px',
    border_radius='5px'
))

display(control_panel)